# Type Validation — Strings in Numeric Fields
Detects non-numeric values in columns that should be numeric, and identifies comma decimal separators (Spanish locale).

**Configure the cell below**, then **Run All**. The final cell fixes commas and casts to numeric — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Columns to validate as numeric
NUMERIC_COLUMNS = {{NUMERIC_COLUMNS}}  # e.g., ["precio", "cantidad", "importe"]
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")

In [ ]:
print("=" * 60)
print("TYPE VALIDATION — Strings in Numeric Fields")
print("=" * 60)

issues = []
comma_columns = []

for col_name in NUMERIC_COLUMNS:
    field = [f for f in df.schema.fields if f.name == col_name][0]

    if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType)):
        print(f"\n✓ '{col_name}' is already {field.dataType} — no string contamination possible.")
        continue

    if isinstance(field.dataType, StringType):
        # Check for non-numeric values
        cast_check = df.where(F.col(col_name).isNotNull()).withColumn(
            "_numeric_cast", F.col(col_name).cast(DoubleType())
        )
        bad_rows = cast_check.where(
            F.col("_numeric_cast").isNull() & F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) != "")
        )
        bad_count = bad_rows.count()

        if bad_count > 0:
            print(f"\n⚠ '{col_name}': {bad_count:,} values cannot be parsed as numeric:")
            display(bad_rows.select(col_name).distinct().limit(20))
            issues.append((col_name, bad_count))
        else:
            print(f"\n✓ '{col_name}' (StringType) — all non-null values are valid numbers.")

        # Check for comma decimal separators (Spanish locale)
        comma_decimal = df.where(F.col(col_name).rlike(r"^\d+,\d+$")).count()
        if comma_decimal > 0:
            print(f"  ⚠ '{col_name}': {comma_decimal:,} values use comma as decimal separator")
            comma_columns.append(col_name)

print(f"\n── Summary ──")
print(f"Columns with non-numeric strings: {len(issues)}")
print(f"Columns with comma decimals: {len(comma_columns)}")

---
## ⚠ Apply Fix — Fix Commas & Cast to Numeric
The cell below replaces comma decimal separators with dots and casts string columns to DoubleType.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df

for col_name in comma_columns:
    cleaned_df = cleaned_df.withColumn(
        col_name,
        F.regexp_replace(F.col(col_name), r"^(\d+),(\d+)$", "$1.$2")
    )
print(f"Fixed comma decimal separators in {len(comma_columns)} columns")

for col_name in NUMERIC_COLUMNS:
    field = [f for f in cleaned_df.schema.fields if f.name == col_name][0]
    if isinstance(field.dataType, StringType):
        cleaned_df = cleaned_df.withColumn(col_name, F.col(col_name).cast(DoubleType()))
print(f"Cast {len(NUMERIC_COLUMNS)} columns to DoubleType")

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Output: {output_table}")